# Script To process EAWAG and KIST RBR CTD Data (files.dat)

In [1]:
import os
import glob
import pandas as pd
import numpy as np
import shutil
import re
from datetime import datetime

In [2]:
def read_rbr_dat(
    filepath,
    recover_unnamed_depth=True,
    return_info=False
):
    """
    Read an original raw RBR XR-620 .dat file.

    Supports files with and without an existing sample-index column.

    Optional:
    - recover_unnamed_depth:
        If the header contains Cond, Temp and Pres but no Depth,
        one additional unnamed numerical column is interpreted as Depth.

    - return_info:
        If True, return (df, info), where info describes any adjustment.

    Returns
    -------
    df : pandas.DataFrame
        Numerical measurement data.

    info : dict, optional
        Information about header adjustments, returned only when
        return_info=True.
    """

    filepath = os.fspath(filepath)

    info = {
        "depth_recovered": False,
        "adjustment": None
    }

    with open(filepath, "r", encoding="utf-8", errors="replace") as f:
        lines = f.readlines()

    # ---------------------------------------------------------
    # 1. Find the measurement header dynamically
    # ---------------------------------------------------------
    header_idx = None
    column_names = None
    partial_header_idx = None

    required_columns = ["Cond", "Temp", "Pres"]
    expected_columns = required_columns + ["Depth"]

    for i, line in enumerate(lines):

        tokens = line.strip().split()

        if all(col in tokens for col in expected_columns):
            header_idx = i
            column_names = tokens
            break

        elif all(col in tokens for col in required_columns):
            partial_header_idx = i

    if header_idx is None:

        if partial_header_idx is not None:

            if not recover_unnamed_depth:
                raise ValueError(
                    "MISSING DEPTH COLUMN: "
                    "Measurement header contains Cond, Temp and Pres, "
                    "but Depth is missing."
                )

            header_idx = partial_header_idx
            column_names = lines[header_idx].strip().split()

            # Mark that Depth recovery needs to be checked
            info["depth_recovery_pending"] = True

        else:

            raise ValueError(
                "MISSING MEASUREMENT HEADERS: "
                "No identifiable column-header row was found."
            )

    else:
        info["depth_recovery_pending"] = False

    # ---------------------------------------------------------
    # 2. Read numerical measurement data
    # ---------------------------------------------------------
    data_rows = []

    for line in lines[header_idx + 1:]:

        tokens = line.strip().split()

        if not tokens:
            continue

        try:
            values = [float(value) for value in tokens]
        except ValueError:
            continue

        data_rows.append(values)

    if not data_rows:
        raise ValueError("No numerical measurement data found.")

    # ---------------------------------------------------------
    # 3. Determine the number of columns
    # ---------------------------------------------------------
    ncols = max(len(row) for row in data_rows)
    n_header = len(column_names)

    # ---------------------------------------------------------
    # 4. Detect an existing sample-index column
    # ---------------------------------------------------------
    has_existing_indices = False

    if ncols > n_header:

        first_values = np.array(
            [row[0] for row in data_rows],
            dtype=float
        )

        expected_indices = np.arange(1, len(data_rows) + 1)

        has_existing_indices = np.allclose(
            first_values,
            expected_indices
        )

    # ---------------------------------------------------------
    # 5. Assign column names
    # ---------------------------------------------------------
    column_names = list(column_names)

    if has_existing_indices:

        # The first numerical column contains original sample indices.
        column_names.insert(0, "Datasets")

    # ---------------------------------------------------------
    # 5.1 Recover missing Depth column if requested
    # ---------------------------------------------------------
    if info.get("depth_recovery_pending", False):

        expected_ncols = len(column_names) + 1

        if ncols == expected_ncols:

            # Insert Depth immediately after Pres.
            pres_position = column_names.index("Pres")

            column_names.insert(
                pres_position + 1,
                "Depth"
            )

            info["depth_recovered"] = True
            info["adjustment"] = (
                "Unnamed numerical column assigned as Depth."
            )

        elif ncols > expected_ncols:

            raise ValueError(
                "AMBIGUOUS UNNAMED COLUMNS: "
                "More than one unnamed numerical column was found "
                "when attempting to recover Depth."
            )

        else:

            raise ValueError(
                "MISSING DEPTH COLUMN: "
                "No additional numerical column was available "
                "to recover Depth."
            )

    # Add names for additional unnamed columns.
    while len(column_names) < ncols:

        extra_number = len(column_names) - n_header + 1
        column_names.append(f"Extra_{extra_number}")

    # Check for inconsistent row lengths.
    if any(len(row) > len(column_names) for row in data_rows):
        raise ValueError(
            "Some measurement rows contain more values "
            "than the identified column structure."
        )

    # Pad rows with missing values where necessary.
    data_rows = [
        row + [np.nan] * (ncols - len(row))
        for row in data_rows
    ]

    # ---------------------------------------------------------
    # 6. Construct the DataFrame
    # ---------------------------------------------------------
    df = pd.DataFrame(data_rows, columns=column_names)

    # Create sample indices only when they were not present.
    if not has_existing_indices:

        df.insert(
            0,
            "Datasets",
            np.arange(1, len(df) + 1)
        )

    if return_info:
        return df, info

    return df


def separate_rbr_profiles(
    df,
    depth_column="Depth",
    min_descent_depth=2.0,
    surface_search_depth=0.5,
):
    """
    Separate multiple CTD profiles from an RBR data file using depth.

    The input DataFrame is expected to be returned by read_rbr_dat(),
    which handles sample indices and column naming.

    Original sample indices in the Datasets column are preserved
    when extracting individual profiles.

    Parameters
    ----------
    df : pandas.DataFrame
        DataFrame containing the RBR measurements.
    depth_column : str
        Name of the depth column.
    min_descent_depth : float
        Minimum depth (m) that must be exceeded before a subsequent
        surface return can be considered a new profile.
    surface_search_depth : float
        Upper depth threshold (m) defining the near-surface region.
        Measurements satisfying Depth <= surface_search_depth are
        considered part of the surface-return interval.

    Returns
    -------
    profiles : list of pandas.DataFrame
        Individual CTD profiles.
    boundaries : pandas.DataFrame
        Summary of the detected profile boundaries.
    """

    # ---------------------------------------------------------
    # 1. Validate input
    # ---------------------------------------------------------
    if df.empty:
        raise ValueError("The input DataFrame is empty.")

    if depth_column not in df.columns:
        raise ValueError(
            f"Column '{depth_column}' not found in df. "
            f"Available columns: {df.columns.tolist()}"
        )

    if min_descent_depth <= surface_search_depth:
        raise ValueError(
            "min_descent_depth must be greater than surface_search_depth."
        )

    # ---------------------------------------------------------
    # 2. Extract depth measurements
    # ---------------------------------------------------------
    depth = pd.to_numeric(
        df[depth_column], errors="coerce"
    ).to_numpy(dtype=float)

    n_samples = len(depth)

    if not np.isfinite(depth).any():
        raise ValueError(
            f"Column '{depth_column}' contains no valid depth values."
        )

    # ---------------------------------------------------------
    # 3. Detect profile starts
    # ---------------------------------------------------------
    # The first profile always starts at the first measurement.
    profile_starts = [0]

    # State variables
    has_descended = False
    pending_candidate = None

    i = 0

    while i < n_samples:

        current_depth = depth[i]

        if not np.isfinite(current_depth):
            i += 1
            continue

        # -----------------------------------------------------
        # 3.1 Confirm a candidate surface return after another
        #     sufficiently deep descent.
        # -----------------------------------------------------
        if pending_candidate is not None:

            if current_depth > min_descent_depth:

                # Confirm the new profile boundary.
                if pending_candidate > profile_starts[-1]:
                    profile_starts.append(pending_candidate)

                pending_candidate = None

                # The new profile has already reached the required depth.
                has_descended = True

            i += 1
            continue

        # -----------------------------------------------------
        # 3.2 Wait until the instrument descends sufficiently deep.
        # -----------------------------------------------------
        if not has_descended:

            if current_depth > min_descent_depth:
                has_descended = True

            i += 1
            continue

        # -----------------------------------------------------
        # 3.3 After descent, identify a return to the surface.
        # -----------------------------------------------------
        if current_depth <= surface_search_depth:

            surface_indices = []

            # Collect the continuous near-surface interval.
            while i < n_samples:

                d = depth[i]

                if not np.isfinite(d) or d > surface_search_depth:
                    break

                surface_indices.append(i)
                i += 1

            # Select the measurement closest to zero depth.
            surface_depths = np.abs(depth[surface_indices])

            candidate_position = np.argmin(surface_depths)

            pending_candidate = surface_indices[candidate_position]

            # Wait for a subsequent descent to confirm this candidate.
            has_descended = False

            continue

        i += 1

    # ---------------------------------------------------------
    # 4. Construct individual profiles and boundary information
    # ---------------------------------------------------------
    profiles = []
    boundary_records = []

    profile_ends = profile_starts[1:] + [n_samples]

    for profile_number, (start, end) in enumerate(
        zip(profile_starts, profile_ends), start=1
    ):

        # Preserve all original columns and sample indices.
        profile = df.iloc[start:end].copy()

        profiles.append(profile)

        boundary_records.append({
            "profile": profile_number,
            "start_index": start,
            "end_index": end - 1,
            "n_samples": end - start,
            "start_depth": depth[start],
            "end_depth": depth[end - 1],
        })

    boundaries = pd.DataFrame(boundary_records)

    return profiles, boundaries


def save_rbr_processed(
    infile,
    metadata,
    df,
    output_file=None,
    include_profile_indices=False,
    boundaries=None,
):
    """
    Save a complete processed RBR file containing metadata,
    sample indices, measurements and reconstructed time.

    Optionally includes the start and end sample indices of
    the individual profiles in the metadata.

    Parameters
    ----------
    infile : str
        Original RBR .dat file.
    metadata : dict
        Metadata returned by read_meta_RBR().
    df : pandas.DataFrame
        Processed RBR DataFrame containing Datasets and time.
    output_file : str, optional
        Output file path. If None, '_processed' is appended
        to the original filename.
    include_profile_indices : bool
        Whether to include profile start and end indices
        in the metadata.
    boundaries : pandas.DataFrame, optional
        Profile boundary summary returned by separate_rbr_profiles().

    Returns
    -------
    output_file : str
        Path to the saved file.
    """

    # --------------------------------------------------
    # 1. Define output filename
    # --------------------------------------------------
    if output_file is None:

        folder = os.path.dirname(infile)
        filename = os.path.basename(infile)
        name, ext = os.path.splitext(filename)

        output_file = os.path.join(
            folder, f"{name}_processed{ext}"
        )

    # --------------------------------------------------
    # 2. Define metadata fields and their order
    # --------------------------------------------------
    metadata_fields = [
        "Campaign_number",
        "Profile_count",
        "Profile",
        "date",
        "Latitude_S_(digital)",
        "Longitude_E_(digital)",
        "Distance_to_GEF_(m)",
        "Rope_length_(m)",
        "Max_depth_(m)",
        "TOB_name_in_Database",
        "Purpose_of_sampling",
        "pH_Calibration_(7)",
        "pH_Calibration_(9)",
        "pH_Calibration_(10)",
        "pH_Calibration_(4)",
        "Starting_time",
        "Frequency",
    ]

    # --------------------------------------------------
    # 3. Extract profile indices if requested
    # --------------------------------------------------
    if include_profile_indices:

        if boundaries is None:
            raise ValueError(
                "Profile boundaries must be provided when "
                "include_profile_indices=True."
            )

        if "Datasets" not in df.columns:
            raise ValueError(
                "The DataFrame must contain the 'Datasets' column."
            )

        # Convert zero-based DataFrame positions into
        # the corresponding 1-based Datasets values.
        start_profile_indices = (
            df["Datasets"].iloc[boundaries["start_index"]]
            .astype(int).tolist()
        )

        end_profile_indices = (
            df["Datasets"].iloc[boundaries["end_index"]]
            .astype(int).tolist()
        )

    # --------------------------------------------------
    # 4. Write metadata and measurements
    # --------------------------------------------------
    dict_meta = {}

    for key in metadata_fields:
        if key in metadata:
            dict_meta[f"{key}:"] = metadata[key]

    quote = str(
        pd.DataFrame(dict_meta, index=[0]).iloc[0].to_string()
    )

    with open(output_file, "w", encoding="utf-8") as f:

        f.write("*** Meta Data ***\n")
        f.write(quote)
        f.write("\n\n")
        f.write("*************\n")

        df.to_csv(
            f,
            sep="\t",
            index=False,
            float_format="%.6f",
            date_format="%Y-%m-%d %H:%M:%S.%f",
            lineterminator="\n"
        )

    return output_file

In [3]:

def find_rbr_files(input_folder):
    """
    Find original RBR .dat files.

    For the 2012 dataset, filenames correspond directly to waypoint
    names in the associated waypoint .txt file.

    Example:
        jan 11 01.dat
        jan 11 02.dat

    Returns
    -------
    files : list of dict
        Each entry contains the file path, filename and filename stem.
    """

    files = []

    dat_files = sorted(
        glob.glob(os.path.join(input_folder, "*.dat"))
    )

    for infile in dat_files:

        filename = os.path.basename(infile)
        file_stem = os.path.splitext(filename)[0].strip()

        files.append({
            "infile": infile,
            "filename": filename,
            "stem": file_stem
        })

    if not files:
        raise ValueError(
            f"No RBR .dat files found in {input_folder}."
        )

    # Check for duplicate waypoint names
    stems = [item["stem"].casefold() for item in files]

    if len(stems) != len(set(stems)):
        raise ValueError(
            "Duplicate RBR filenames detected. "
            "Each waypoint name must identify only one .dat file."
        )

    return files


def normalize_rbr_filename(filename):
    """
    Normalise a filename for matching Excel Profile entries with .dat files.

    Spaces and underscores are treated as equivalent.
    File extensions are ignored.
    Matching is case-insensitive.
    """
    filename = os.path.basename(str(filename).strip())
    filename = os.path.splitext(filename)[0]

    return re.sub(r"[\s_]+", "", filename).lower()


def prepare_rbr_metadata(filepath, year=None, institute="EAWAG"):

    df = pd.read_excel(filepath)

    # Standardise column names
    df.columns = df.columns.str.strip()

    required_columns = [
        "Profile", "Code", "Year", "Month", "Day",
        "Hour", "Minute", "North", "East", "Institute"
    ]

    missing_columns = [
        col for col in required_columns
        if col not in df.columns
    ]

    if missing_columns:
        raise ValueError(
            f"Missing required Excel columns: {missing_columns}"
        )

    # Filter by institute
    df = df[
        df["Institute"].astype(str).str.strip().str.upper()
        == institute.upper()
    ].copy()

    # Optional year filter
    if year is not None:
        df["Year"] = pd.to_numeric(df["Year"], errors="coerce")
        df = df[df["Year"] == year].copy()

    # Continue with Profile cleaning and filename normalisation
    df["Profile"] = df["Profile"].astype("string").str.strip()

    df = df[
        df["Profile"].notna() & (df["Profile"] != "")
    ].copy()

    df["Profile_normalized"] = df["Profile"].apply(
        normalize_rbr_filename
    )

    # Convert numerical fields
    numeric_columns = [
        "Year", "Month", "Day", "Hour", "Minute",
        "North", "East"
    ]

    for col in numeric_columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    # Identify invalid records ONLY after institution/year filtering
    invalid_columns = [
        "Year", "Month", "Day", "Hour", "Minute",
        "North", "East"
    ]

    invalid_records = df[
        df[invalid_columns].isna().any(axis=1)
    ].copy()

    # Retain valid records
    df = df.dropna(subset=invalid_columns).copy()

    # Build profile start datetime
    df["start_datetime"] = pd.to_datetime(
        {
            "year": df["Year"].astype(int),
            "month": df["Month"].astype(int),
            "day": df["Day"].astype(int),
            "hour": df["Hour"].astype(int),
            "minute": df["Minute"].astype(int),
            "second": 0
        },
        errors="raise"
    )

    # Clean Code column
    df["Code"] = df["Code"].replace(
        r"^\s*$", np.nan, regex=True
    )

    # Profile_Number
    if "Profile_Number" not in df.columns:

        df["Profile_Number"] = (
            df.groupby("Profile_normalized", sort=False)
              .cumcount() + 1
        )

    else:

        df["Profile_Number"] = pd.to_numeric(
            df["Profile_Number"], errors="coerce"
        )

        for group_name, group in df.groupby(
            "Profile_normalized", sort=False
        ):

            missing_indices = group[
                group["Profile_Number"].isna()
            ].index

            if len(missing_indices) > 0:

                existing_numbers = group[
                    group["Profile_Number"].notna()
                ]["Profile_Number"]

                next_number = (
                    int(existing_numbers.max()) + 1
                    if not existing_numbers.empty
                    else 1
                )

                df.loc[missing_indices, "Profile_Number"] = range(
                    next_number,
                    next_number + len(missing_indices)
                )

    df["Profile_Number"] = df["Profile_Number"].astype(int)

    if (df["Profile_Number"] < 1).any():
        raise ValueError("Profile_Number must be >= 1.")

    duplicates = df.duplicated(
        ["Profile_normalized", "Profile_Number"],
        keep=False
    )

    if duplicates.any():
        raise ValueError(
            "Duplicate Profile_Number values for the same source file:\n"
            f"{df.loc[duplicates, ['Profile', 'Profile_Number']]}"
        )

    # Retain existing processing defaults
    df["Campaign"] = 1
    df["Max_depth"] = np.nan
    df["Frequency"] = 6.0

    df = df.sort_values(
        ["Profile_normalized", "Profile_Number"]
    ).reset_index(drop=True)

    return df, invalid_records


def validate_rbr_profiles(
    df_metadata,
    file_stem,
    n_detected_profiles
):

    normalized_stem = normalize_rbr_filename(file_stem)

    metadata_records = df_metadata[
        df_metadata["Profile_normalized"] == normalized_stem
    ].copy()

    if metadata_records.empty:
        raise ValueError(
            f"No matching metadata found for file: {file_stem}"
        )

    metadata_records = metadata_records.sort_values(
        "Profile_Number"
    ).reset_index(drop=True)

    n_metadata_profiles = len(metadata_records)

    if n_metadata_profiles != n_detected_profiles:
        raise ValueError(
            f"Profile count mismatch for {file_stem}: "
            f"{n_detected_profiles} profiles detected in the file, "
            f"but {n_metadata_profiles} metadata records found."
        )

    return metadata_records


def extract_profile_metadata(row, profile_count, stem):

    start_datetime = row["start_datetime"]

    # Determine output filename
    code = row.get("Code", np.nan)

    # Replace spaces with underscores in the source Profile name
    profile_name = re.sub(
        r"\s+", "_", str(row["Profile"]).strip()
    )

    if pd.notna(code) and str(code).strip():

        code_name = os.path.splitext(
            os.path.basename(str(code).strip())
        )[0]

        tob_name = f"{profile_name}_{code_name}.dat"

    elif profile_count > 1:

        profile_number = int(row["Profile_Number"])
        tob_name = f"{profile_name}_{profile_number}.dat"

    else:

        tob_name = f"{profile_name}.dat"

    metadata = {
        "Campaign_number": int(row["Campaign"]),
        "Profile_count": int(profile_count),
        "Profile": int(row["Profile_Number"]),
        "date": start_datetime.strftime("%y%m%d"),
        "Latitude_S_(digital)": float(row["North"]),
        "Longitude_E_(digital)": float(row["East"]),
        "Distance_to_GEF_(m)": np.nan,
        "Rope_length_(m)": np.nan,
        "Max_depth_(m)": np.nan,
        "TOB_name_in_Database": tob_name,
        "Purpose_of_sampling": row.get("comments", np.nan),
        "pH_Calibration_(7)": np.nan,
        "pH_Calibration_(9)": np.nan,
        "pH_Calibration_(10)": np.nan,
        "pH_Calibration_(4)": np.nan,
        "Starting_time": start_datetime.strftime("%H:%M:%S"),
        "Frequency": float(row["Frequency"]),
        "start_datetime": start_datetime
    }

    return metadata


def process_rbr_file(
    infile,
    df_waypoints,
    output_folder,
    campaign=1
):
    """
    Process one original RBR .dat file:
      1. Read measurements.
      2. Separate individual profiles.
      3. Validate profile count against Excel records.
      4. Extract profile-specific metadata.
      5. Reconstruct timestamps.
      6. Save each profile independently.
    """

    filename = os.path.basename(infile)
    stem = os.path.splitext(filename)[0]

    print(f"\nProcessing {filename}")

    # --------------------------------------------------
    # 1. Extract date from filename
    # --------------------------------------------------
    date_string = stem.split("_")[-1]

    file_date = pd.to_datetime(
        date_string,
        format="%y%m%d",
        errors="raise"
    ).normalize()

    # --------------------------------------------------
    # 2. Read original measurements
    # --------------------------------------------------
    df = read_rbr_dat(infile)

    # Retain original sample numbering
    if "Datasets" not in df.columns:
        df.insert(
            0,
            "Datasets",
            np.arange(1, len(df) + 1)
        )

    # --------------------------------------------------
    # 3. Separate individual profiles
    # --------------------------------------------------
    profiles, boundaries = separate_rbr_profiles(df)

    n_profiles = len(profiles)

    if n_profiles == 0:
        raise ValueError(
            f"No profiles detected in {filename}."
        )

    print(f"Detected profiles: {n_profiles}")

    # --------------------------------------------------
    # 4. Validate against Excel waypoint records
    # --------------------------------------------------
    waypoints = validate_rbr_profiles(
        df_waypoints=df_waypoints,
        file_date=file_date,
        n_detected_profiles=n_profiles,
        campaign=campaign
    )

    print("Profile count validation successful.")

    # --------------------------------------------------
    # 5. Process and save individual profiles
    # --------------------------------------------------
    saved_files = []

    for i, profile_df in enumerate(profiles):

        # Corresponding Excel record
        waypoint = waypoints.iloc[i]

        profile_number = int(waypoint["Profile"])

        print(
            f"  Processing profile {profile_number}/"
            f"{n_profiles}"
        )

        # Extract profile-specific metadata
        metadata = extract_profile_metadata(
            waypoint,
            profile_count=n_profiles,
            stem=stem
        )

        # Reconstruct time using Excel date, time and frequency
        profile_df = add_rbr_time(profile_df, metadata)
        # Rename columns to match the CTD database convention
        profile_df = rename_rbr_columns(profile_df)

        profile_df = profile_df.reset_index(drop=True)

        # Local boundaries for the individual profile
        profile_boundaries = pd.DataFrame({
            "profile": [profile_number],
            "start_index": [0],
            "end_index": [len(profile_df) - 1],
            "n_samples": [len(profile_df)]
        })

        # Output filename: original stem + profile number
        output_file = os.path.join(
            output_folder,
            f"{stem}_{profile_number}.dat"
        )

        # Save individual profile
        saved_file = save_rbr_processed(
            infile=infile,
            metadata=metadata,
            df=profile_df,
            output_file=output_file,
            include_profile_indices=True,
            boundaries=profile_boundaries
        )

        saved_files.append(saved_file)

    print(f"Saved {len(saved_files)} profiles from {filename}.")

    return saved_files

def add_rbr_time(df, metadata):
    """
    Add reconstructed timestamps using profile-specific
    date, time and sampling frequency.
    """

    df = df.copy()

    if "Datasets" not in df.columns:
        df.insert(
            0,
            "Datasets",
            np.arange(1, len(df) + 1)
        )

    start_datetime = metadata["start_datetime"]
    frequency = metadata["Frequency"]

    if not np.isfinite(frequency) or frequency <= 0:
        raise ValueError(
            "Invalid or missing sampling frequency."
        )

    time_seconds = np.arange(len(df)) / frequency

    df["time"] = (
        start_datetime +
        pd.to_timedelta(time_seconds, unit="s")
    )

    return df

def rename_rbr_columns(df):
    column_map = {
        "Pres": "Press"
    }

    df = df.rename(columns=column_map)

    return df

def process_all_rbr_files(
    input_folder,
    df_waypoints,
    campaign=1
):
    """
    Process all original RBR .dat files in a directory.

    The output directory 'rbr_db_data' is deleted if it
    already exists, then recreated before processing.

    Returns
    -------
    all_saved_files : list
        Paths to all successfully saved profile files.
    """

    # --------------------------------------------------
    # 1. Identify original RBR files
    # --------------------------------------------------
    files = find_rbr_files(input_folder)

    print(f"Found {len(files)} original RBR files.")

    # --------------------------------------------------
    # 2. Create a fresh output directory
    # --------------------------------------------------
    output_folder = os.path.join(
        input_folder,
        "rbr_db_data"
    )

    if os.path.exists(output_folder):
        shutil.rmtree(output_folder)

    os.makedirs(output_folder)

    print(f"Output directory: {output_folder}")

    # --------------------------------------------------
    # 3. Process all files
    # --------------------------------------------------
    all_saved_files = []
    failed_files = []

    for item in files:

        infile = item["infile"]

        try:
            saved_files = process_rbr_file(
                infile=infile,
                df_waypoints=df_waypoints,
                output_folder=output_folder,
                campaign=campaign
            )

            all_saved_files.extend(saved_files)

        except Exception as e:

            print(
                f"\nERROR processing {item['filename']}:\n{e}"
            )

            failed_files.append({
                "filename": item["filename"],
                "error": str(e)
            })

    # --------------------------------------------------
    # 4. Report results
    # --------------------------------------------------
    print("\n" + "=" * 50)
    print("RBR PROCESSING SUMMARY")
    print("=" * 50)

    print(f"Original files found: {len(files)}")
    print(f"Profiles successfully saved: {len(all_saved_files)}")
    print(f"Files with errors: {len(failed_files)}")

    if failed_files:
        print("\nFiles requiring attention:")

        for item in failed_files:
            print(f"  {item['filename']}: {item['error']}")

    return all_saved_files, failed_files

In [4]:
# Extra processing to deal with files with multiple profiles and single metdata record

def normalize_source_filename(filename):
    """
    Normalise a source filename for matching.

    Differences in case, spaces, underscores and file extensions
    are ignored. Other characters and digits are retained.
    """
    if pd.isna(filename) or not str(filename).strip():
        return None

    filename = os.path.basename(str(filename).strip())
    filename = os.path.splitext(filename)[0]
    filename = re.sub(r"[\s_]+", "", filename)

    return filename.lower()


def extract_extra_filename(date_filename):
    """
    Extract the date/time and source filename from the
    extra metadata 'Date Filename' field.

    Expected format:
        Oct 28 12:34 018578_water_sampling_101028

    Returns
    -------
    date_part : datetime
        Parsed month, day and time. The year is not used here.
    source_filename : str
        Filename portion following the date/time.
    """
    if pd.isna(date_filename):
        raise ValueError("Date Filename is empty.")

    value = str(date_filename).strip()

    pattern = (
        r"^([A-Za-z]{3}\s+\d{1,2}\s+\d{1,2}:\d{2})\s+(.+)$"
    )

    match = re.match(pattern, value)

    if match is None:
        raise ValueError(
            f"Cannot interpret Date Filename: {value!r}. "
            "Expected format such as 'Oct 28 12:34 filename'."
        )

    date_string = match.group(1).strip()
    source_filename = match.group(2).strip()

    try:
        date_part = datetime.strptime(date_string, "%b %d %H:%M")
    except ValueError as error:
        raise ValueError(
            f"Invalid date/time in Date Filename: {value!r}"
        ) from error

    if not source_filename:
        raise ValueError(
            f"No source filename found in Date Filename: {value!r}"
        )

    return date_part, source_filename


def prepare_extra_metadata(filepath):

    df = pd.read_excel(filepath)

    df.columns = df.columns.astype(str).str.strip()

    # Allow alternative column names
    column_mapping = {
        "profile number": "Profile number",
        "profile_number": "Profile number",
        "date filename": "Date Filename",
        "date_filename": "Date Filename",
        "x": "x",
        "y": "y",
    }

    rename_columns = {}

    for column in df.columns:

        normalised = column.strip().lower()

        if normalised in column_mapping:
            rename_columns[column] = column_mapping[normalised]

    df = df.rename(columns=rename_columns)

    # Campaign is deliberately excluded.
    required_columns = [
        "Profile number",
        "Date Filename",
        "x",
        "y",
    ]

    missing_columns = [
        column for column in required_columns
        if column not in df.columns
    ]

    if missing_columns:
        raise ValueError(
            "Missing required columns in extra metadata: "
            f"{missing_columns}. "
            f"Available columns: {df.columns.tolist()}"
        )

    df = df.copy()

    df["Profile number"] = pd.to_numeric(
        df["Profile number"], errors="coerce"
    )

    df["x"] = pd.to_numeric(df["x"], errors="coerce")
    df["y"] = pd.to_numeric(df["y"], errors="coerce")

    parsed_dates = []
    source_filenames = []
    parse_errors = []

    for value in df["Date Filename"]:

        try:
            date_part, source_filename = extract_extra_filename(value)

            parsed_dates.append(date_part)
            source_filenames.append(source_filename)
            parse_errors.append(None)

        except (ValueError, TypeError) as error:

            parsed_dates.append(pd.NaT)
            source_filenames.append(None)
            parse_errors.append(str(error))

    df["Date_part"] = parsed_dates
    df["Source_Filename"] = source_filenames
    df["Parse_error"] = parse_errors

    df["Source_Filename_normalized"] = (
        df["Source_Filename"].apply(normalize_source_filename)
    )

    return df


def recover_extra_profile_metadata(
    main_records,
    df_extra_metadata,
    n_detected_profiles,
):
    """
    Recover missing profile metadata from the additional metadata file.

    Matching logic:
        Original metadata 'Filename'
            is compared with the filename portion of
        Extra metadata 'Date Filename'.

    The original metadata 'Year' is used for all recovered dates.

    Coordinates:
        Extra metadata x -> East (longitude)
        Extra metadata y -> North (latitude)

    Returns
    -------
    pd.DataFrame
        Reconstructed metadata records, with the same main fields
        expected by extract_profile_metadata().
    """

    if main_records.empty:
        raise ValueError(
            "Cannot recover metadata: no matching original metadata "
            "records were found for this .dat file."
        )

    # ---------------------------------------------------------
    # 1. Obtain the source filename from the original metadata.
    # ---------------------------------------------------------
    if "Filename" not in main_records.columns:
        raise ValueError(
            "The original metadata does not contain a 'Filename' column. "
            "Extra metadata recovery cannot be performed."
        )

    source_filenames = (
        main_records["Filename"]
        .dropna()
        .astype(str)
        .str.strip()
    )

    source_filenames = source_filenames[
        source_filenames != ""
    ]

    if source_filenames.empty:
        raise ValueError(
            "Extra metadata recovery skipped: the original metadata "
            "'Filename' field is empty."
        )

    normalized_filenames = {
        normalize_source_filename(value)
        for value in source_filenames
    }

    normalized_filenames.discard(None)

    if len(normalized_filenames) != 1:
        raise ValueError(
            "Ambiguous source filename in original metadata. "
            f"Found multiple identifiers: {sorted(normalized_filenames)}"
        )

    source_filename_normalized = next(iter(normalized_filenames))

    # ---------------------------------------------------------
    # 2. Obtain the year from the original metadata.
    # ---------------------------------------------------------
    if "Year" not in main_records.columns:
        raise ValueError(
            "The original metadata does not contain a 'Year' column."
        )

    years = pd.to_numeric(
        main_records["Year"], errors="coerce"
    ).dropna().unique()

    if len(years) != 1:
        raise ValueError(
            "Cannot determine a unique year from the original metadata. "
            f"Found: {years.tolist()}"
        )

    year = int(years[0])

    # ---------------------------------------------------------
    # 3. Find matching records in the extra metadata.
    # ---------------------------------------------------------
    extra_records = df_extra_metadata[
        df_extra_metadata["Source_Filename_normalized"]
        == source_filename_normalized
    ].copy()

    if extra_records.empty:
        raise ValueError(
            "No matching records found in extra metadata for source "
            f"filename: {source_filename_normalized!r}."
        )

    # ---------------------------------------------------------
    # 4. Validate the number of extra metadata records.
    # ---------------------------------------------------------
    if len(extra_records) != n_detected_profiles:
        raise ValueError(
            "Extra metadata profile count mismatch: "
            f"{n_detected_profiles} profiles detected in the .dat file, "
            f"but {len(extra_records)} matching records found in "
            "extra metadata."
        )

    # ---------------------------------------------------------
    # 5. Validate profile numbers.
    # ---------------------------------------------------------
    if extra_records["Profile number"].isna().any():
        raise ValueError(
            "Extra metadata contains missing or invalid profile numbers."
        )

    if not np.all(
        extra_records["Profile number"]
        == extra_records["Profile number"].astype(int)
    ):
        raise ValueError(
            "Extra metadata contains non-integer profile numbers."
        )

    extra_records["Profile number"] = (
        extra_records["Profile number"].astype(int)
    )

    expected_profile_numbers = set(
        range(1, n_detected_profiles + 1)
    )

    actual_profile_numbers = set(
        extra_records["Profile number"]
    )

    if (
        len(actual_profile_numbers) != n_detected_profiles
        or actual_profile_numbers != expected_profile_numbers
    ):
        raise ValueError(
            "Invalid profile numbering in extra metadata. "
            f"Expected {sorted(expected_profile_numbers)}, "
            f"found {sorted(actual_profile_numbers)}."
        )

    # Check geographical coordinate ranges.
    # x is interpreted as longitude; y as latitude.
    if not extra_records["x"].between(-180, 180).all():
        raise ValueError(
            "Invalid longitude values in extra metadata (x)."
        )

    if not extra_records["y"].between(-90, 90).all():
        raise ValueError(
            "Invalid latitude values in extra metadata (y)."
        )

    # ---------------------------------------------------------
    # 7. Validate and construct the recovered timestamps.
    # ---------------------------------------------------------
    recovered_records = extra_records.copy()

    recovered_records["start_datetime"] = recovered_records[
        "Date_part"
    ].apply(
        lambda value: datetime(
            year=year,
            month=value.month,
            day=value.day,
            hour=value.hour,
            minute=value.minute,
            second=0,
        )
    )

    # ---------------------------------------------------------
    # 8. Build records compatible with the existing framework.
    # ---------------------------------------------------------
    recovered_records = recovered_records.sort_values(
        "Profile number"
    ).reset_index(drop=True)

    # Map original metadata fields to the reconstructed records.
    # The original Profile identifies the actual .dat file.
    original_profile = main_records.iloc[0]["Profile"]

    recovered_records["Profile"] = original_profile
    recovered_records["Filename"] = (
        main_records.iloc[0]["Filename"]
    )

    recovered_records["Profile_Number"] = (
        recovered_records["Profile number"]
    )

    recovered_records["Year"] = year
    recovered_records["Month"] = recovered_records[
        "start_datetime"
    ].dt.month

    recovered_records["Day"] = recovered_records[
        "start_datetime"
    ].dt.day

    recovered_records["Hour"] = recovered_records[
        "start_datetime"
    ].dt.hour

    recovered_records["Minute"] = recovered_records[
        "start_datetime"
    ].dt.minute

    # Coordinate mapping: x = longitude, y = latitude.
    recovered_records["East"] = recovered_records["x"]
    recovered_records["North"] = recovered_records["y"]

    recovered_records["Campaign"] = 1 # default

    recovered_records["Frequency"] = 6.0
    recovered_records["Max_depth"] = np.nan

    # Preserve the original Code only for the corresponding
    # profile number. Other recovered profiles use the existing
    # profile-number fallback in the output naming convention.
    recovered_records["Code"] = np.nan

    if "Code" in main_records.columns:
        for _, original_row in main_records.iterrows():

            profile_number = original_row.get(
                "Profile_Number", np.nan
            )

            code = original_row.get("Code", np.nan)

            if (
                pd.notna(profile_number)
                and pd.notna(code)
                and str(code).strip()
            ):
                profile_number = int(profile_number)

                recovered_records.loc[
                    recovered_records["Profile_Number"]
                    == profile_number,
                    "Code",
                ] = code

    # Preserve comments where available.
    if "comments" in main_records.columns:
        recovered_records["comments"] = (
            main_records.iloc[0]["comments"]
        )
    else:
        recovered_records["comments"] = np.nan

    # ---------------------------------------------------------
    # 9. Final validation.
    # ---------------------------------------------------------
    if recovered_records["start_datetime"].isna().any():
        raise ValueError(
            "One or more recovered timestamps could not be constructed."
        )

    if len(recovered_records) != n_detected_profiles:
        raise ValueError(
            "Internal recovery error: the number of reconstructed "
            "metadata records does not match the detected profile count."
        )

    return recovered_records

In [8]:
## Running data processing framework function
def process_extra_rbr_data(input_folder, metadata_file, extra_metadata_file, which_institute, year=None):
    # ---------------------------------------------------------
    # 2. Prepare original metadata
    # ---------------------------------------------------------
    institute = which_institute

    df_metadata, invalid_metadata = prepare_rbr_metadata(filepath=metadata_file,year=year, institute=which_institute)


    # ---------------------------------------------------------
    # 3. Identify original RBR files
    # ---------------------------------------------------------

    rbr_files = find_rbr_files(input_folder)


    # ---------------------------------------------------------
    # 4. Create output directory and history file
    # ---------------------------------------------------------

    if year is None:
        output_folder_name = f"Processed_profiles_{institute}"
    else:
        output_folder_name = f"Processed_profiles_{year}"

    #output_folder = os.path.join(input_folder, output_folder_name)
    output_folder = os.path.join(os.path.dirname(os.path.normpath(input_folder)), output_folder_name)

    os.makedirs(output_folder, exist_ok=True)

    history_file = os.path.join(output_folder,"history.txt")


    # ---------------------------------------------------------
    # 5. Initialise processing history
    # ---------------------------------------------------------

    with open(history_file, "w", encoding="utf-8") as f:

        f.write("RBR Processing History\n")
        f.write("=" * 50 + "\n\n")

        if not invalid_metadata.empty:

            f.write("EXCLUDED METADATA RECORDS\n")
            f.write("-" * 50 + "\n")

            for _, row in invalid_metadata.iterrows():

                missing = [
                    col for col in [
                        "Year", "Month", "Day", "Hour", "Minute",
                        "North", "East"
                    ]
                    if pd.isna(row[col])
                ]

                f.write(
                    f"Profile: {row['Profile']}\n"
                    f"Code: {row['Code']}\n"
                    f"Missing fields: {', '.join(missing)}\n"
                    f"{'-' * 50}\n"
                )

            f.write(
                f"\nTotal excluded metadata records: "
                f"{len(invalid_metadata)}\n\n"
            )


    # ---------------------------------------------------------
    # 6. Initialise processing counters
    # ---------------------------------------------------------

    n_processed = 0
    n_skipped = 0
    n_no_metadata = 0

    n_recovered_files = 0
    n_recovered_profiles = 0

    saved_output_names = set()

    # Extra metadata will only be loaded if recovery is needed.
    df_extra_metadata = None


    # ---------------------------------------------------------
    # 7. Process individual RBR files
    # ---------------------------------------------------------

    for file_info in rbr_files:

        infile = file_info["infile"]
        stem = file_info["stem"]
        filename = file_info["filename"]

        print(f"\nProcessing: {filename}")


        try:

            # -------------------------------------------------
            # 7.1 Retrieve original metadata FIRST
            # -------------------------------------------------

            normalized_stem = normalize_rbr_filename(stem)

            main_records = df_metadata[
                df_metadata["Profile_normalized"] == normalized_stem
            ].copy()

            # Skip files without metadata for the selected institution.
            if main_records.empty:

                print(
                    f" SKIPPED: {filename} "
                    f"(no metadata for institute {institute})"
                )

                n_no_metadata += 1
                continue

            main_records = main_records.sort_values(
                "Profile_Number"
            ).reset_index(drop=True)

            n_main_records = len(main_records)

            print(
                f"Original metadata records: {n_main_records}"
            )

            # -------------------------------------------------
            # 7.2 Read original RBR file
            # -------------------------------------------------

            df = read_rbr_dat(infile)

            # -------------------------------------------------
            # 7.3 Separate individual profiles
            # -------------------------------------------------

            profiles, boundaries = separate_rbr_profiles(
                df,
                min_descent_depth=5.0,
                surface_search_depth=0.05
            )

            n_profiles = len(profiles)

            print(f"Detected profiles: {n_profiles}")

            if n_profiles == 0:
                raise ValueError(
                    "No profiles could be detected in the RBR file."
                )

            # -------------------------------------------------
            # 7.4 Check whether extra metadata recovery is needed
            # -------------------------------------------------

            used_recovery = False

            if n_main_records == n_profiles:

                # Original metadata is complete.
                metadata_records = main_records

                print(
                    "Original metadata matches the detected "
                    "number of profiles."
                )

            elif n_main_records < n_profiles:

                # Original metadata contains fewer records
                # than the number of detected profiles.
                print(
                    "Original metadata is incomplete. "
                    "Attempting recovery using extra metadata..."
                )

                # Load extra metadata only when needed.
                if df_extra_metadata is None:

                    if not os.path.isfile(extra_metadata_file):
                        raise FileNotFoundError(
                            "Extra metadata file not found: "
                            f"{extra_metadata_file}"
                        )

                    df_extra_metadata = prepare_extra_metadata(
                        extra_metadata_file
                    )

                # Recover metadata for all detected profiles.
                metadata_records = recover_extra_profile_metadata(
                    main_records=main_records,
                    df_extra_metadata=df_extra_metadata,
                    n_detected_profiles=n_profiles
                )

                used_recovery = True

                print(
                    f"Successfully recovered metadata for "
                    f"{len(metadata_records)} profiles."
                )

            else:

                # More original metadata records than detected profiles.
                raise ValueError(
                    f"Original metadata profile count mismatch: "
                    f"{n_profiles} profiles detected in the file, "
                    f"but {n_main_records} original metadata records found. "
                    "Extra metadata recovery is only intended for "
                    "missing original metadata records."
                )
            
            # -------------------------------------------------
            # 7.5 Validate recovered metadata against duplication
            # -------------------------------------------------

            if used_recovery:

                if len(metadata_records) != n_profiles:
                    raise ValueError(
                        "Recovered metadata contains a different number "
                        "of records than detected profiles."
                    )

                if metadata_records["Profile_Number"].duplicated().any():
                    raise ValueError(
                        "Duplicate profile numbers found in recovered metadata."
                    )

                if metadata_records["Profile_Number"].nunique() != n_profiles:
                    raise ValueError(
                        "Recovered metadata does not contain one unique "
                        "record for each detected profile."
                    )

            # -------------------------------------------------
            # 7.6 Final metadata count and ordering validation
            # -------------------------------------------------

            metadata_records = metadata_records.sort_values(
                "Profile_Number"
            ).reset_index(drop=True)

            if len(metadata_records) != n_profiles:
                raise ValueError(
                    f"Final metadata count mismatch: "
                    f"{n_profiles} profiles detected, "
                    f"but {len(metadata_records)} metadata records available."
                )

            # -------------------------------------------------
            # 7.7 Prepare metadata for all profiles
            # -------------------------------------------------

            profile_metadata = []

            for _, row in metadata_records.iterrows():

                metadata = extract_profile_metadata(
                    row=row,
                    profile_count=n_profiles,
                    stem=stem
                )

                profile_metadata.append(metadata)

            # -------------------------------------------------
            # 7.8 Check for duplicate output filenames
            # -------------------------------------------------

            output_names = [
                item["TOB_name_in_Database"]
                for item in profile_metadata
            ]

            if len(output_names) != len(set(output_names)):
                raise ValueError(
                    f"Duplicate output filenames generated for {filename}: "
                    f"{output_names}"
                )

            # Check collisions with previously processed files.
            previous_collisions = [
                name for name in output_names
                if name in saved_output_names
            ]

            if previous_collisions:
                raise ValueError(
                    f"Output filename already generated by another "
                    f"source file: {previous_collisions}"
                )

            # Also check whether files already exist in the output folder.
            existing_collisions = [
                name for name in output_names
                if os.path.exists(os.path.join(output_folder, name))
            ]

            if existing_collisions:
                raise ValueError(
                    f"Output files already exist in the output folder: "
                    f"{existing_collisions}"
                )

            # -------------------------------------------------
            # 7.9 Prepare all profiles before saving
            # -------------------------------------------------

            prepared_profiles = []

            for profile, metadata in zip(
                profiles,
                profile_metadata
            ):

                profile = add_rbr_time(
                    profile,
                    metadata
                )

                profile = profile.rename(
                    columns={"Pres": "Press"}
                )

                prepared_profiles.append(
                    (profile, metadata)
                )

            # -------------------------------------------------
            # 7.10 Save individual profiles
            # -------------------------------------------------

            for profile, metadata in prepared_profiles:

                output_name = metadata["TOB_name_in_Database"]

                output_file = os.path.join(
                    output_folder,
                    output_name
                )

                save_rbr_processed(
                    infile=infile,
                    metadata=metadata,
                    df=profile,
                    output_file=output_file
                )

                saved_output_names.add(output_name)

                print(f"  Saved: {output_name}")

                n_processed += 1

            # Update recovery counters only after successful saving.
            if used_recovery:

                n_recovered_files += 1
                n_recovered_profiles += len(metadata_records)

        # -------------------------------------------------
        # 7.11 Log successful processing
        # -------------------------------------------------
            
        except Exception as error:

            n_skipped += 1

            error_text = str(error)

            if "No matching metadata found" in error_text:

                n_no_metadata += 1

                print(
                    f"  SKIPPED: {filename} "
                    "(no matching metadata found)"
                )

                continue

            elif "MISSING MEASUREMENT HEADERS" in error_text:

                status = "MISSING MEASUREMENT HEADERS"

            elif "MISSING DEPTH COLUMN" in error_text:

                status = "MISSING DEPTH COLUMN"

            else:

                status = "PROCESSING ERROR"

            error_message = (
                f"{status}\n"
                f"File: {filename}\n"
                f"Reason: {type(error).__name__}: {error}\n"
                f"No profiles from this source file were saved.\n"
                f"{'-' * 50}\n\n"
            )

            with open(history_file, "a", encoding="utf-8") as f:
                f.write(error_message)

            print(f"  SKIPPED: {filename}")
            print(f"  Reason: {error}")        


    # ---------------------------------------------------------
    # 8. Processing summary
    # ---------------------------------------------------------

    print("\n" + "=" * 50)
    print("RBR PROCESSING SUMMARY")
    print("=" * 50)

    print(f"Year filter: {year}")
    print(f"Institution: {institute}")
    print(f"Total original files: {len(rbr_files)}")
    print(f"Profiles successfully saved: {n_processed}")
    print(f"Files skipped: {n_skipped}")
    print(f"Files without matching metadata: {n_no_metadata}")
    print(f"Files recovered using extra metadata: {n_recovered_files}")
    print(f"Profiles recovered using extra metadata: {n_recovered_profiles}")
    print(f"Other processing errors: {n_skipped - n_no_metadata}")

    print(f"\nProcessing history: {history_file}")


    # ---------------------------------------------------------
    # 9. Append processing summary to history
    # ---------------------------------------------------------

    with open(history_file, "a", encoding="utf-8") as f:

        f.write("\n")
        f.write("=" * 50 + "\n")
        f.write("RBR PROCESSING SUMMARY\n")
        f.write("=" * 50 + "\n")

        f.write(f"Year filter: {year}\n")
        f.write(f"Institution: {institute}\n")
        f.write(f"Total original files: {len(rbr_files)}\n")
        f.write(f"Profiles successfully saved: {n_processed}\n")
        f.write(f"Files skipped: {n_skipped}\n")
        f.write(f"Files without matching metadata: {n_no_metadata}\n")
        f.write(f"Files recovered using extra metadata: {n_recovered_files}\n")
        f.write(f"Profiles recovered using extra metadata: {n_recovered_profiles}\n")
        f.write(f"Other processing errors: {n_skipped - n_no_metadata}\n")


In [ ]:
# ---------------------------------------------------------
# 1. Read and prepare the Excel metadata
# ---------------------------------------------------------
input_folder = "Raw_data"
metadata_file = "Metadata/Kivu_CTD_database_ag.xlsx"
extra_metadata_file = "Metadata/Oct_2010_positions.xlsx"
which_institute = "EAWAG" # or KIST
process_extra_rbr_data(input_folder, metadata_file, extra_metadata_file, which_institute, year=None)


Processing: KV_100116_01.dat
 SKIPPED: KV_100116_01.dat (no metadata for institute KIST)

Processing: KV_100117_01.dat
 SKIPPED: KV_100117_01.dat (no metadata for institute KIST)

Processing: KV_100118_01.dat
 SKIPPED: KV_100118_01.dat (no metadata for institute KIST)

Processing: KV_100119_01.dat
 SKIPPED: KV_100119_01.dat (no metadata for institute KIST)

Processing: KV_100120_01.dat
 SKIPPED: KV_100120_01.dat (no metadata for institute KIST)

Processing: KV_100121_01.dat
 SKIPPED: KV_100121_01.dat (no metadata for institute KIST)

Processing: KV_100619_01.dat
Original metadata records: 1
Detected profiles: 2
Original metadata is incomplete. Attempting recovery using extra metadata...
  SKIPPED: KV_100619_01.dat
  Reason: No matching records found in extra metadata for source filename: '1'.

Processing: KV_100619_02.dat
Original metadata records: 1
Detected profiles: 1
Original metadata matches the detected number of profiles.
  Saved: KV_100619_02_Q2.dat

Processing: KV_100619_03.d

/home/mushimje@eawag.wroot.emp-eaw.ch/.conda/envs/kivudatabase/lib/python3.11/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)


  Saved: KV_100619_03_Q3.dat

Processing: KV_101028_01.dat
 SKIPPED: KV_101028_01.dat (no metadata for institute KIST)

Processing: KV_101108_04.dat
 SKIPPED: KV_101108_04.dat (no metadata for institute KIST)

Processing: KV_101108_05.dat
 SKIPPED: KV_101108_05.dat (no metadata for institute KIST)

Processing: KV_101108_06.dat
 SKIPPED: KV_101108_06.dat (no metadata for institute KIST)

Processing: KV_101109_02.dat
 SKIPPED: KV_101109_02.dat (no metadata for institute KIST)

Processing: KV_101109_03.dat
 SKIPPED: KV_101109_03.dat (no metadata for institute KIST)

Processing: KV_101110_01.dat
 SKIPPED: KV_101110_01.dat (no metadata for institute KIST)

Processing: KV_101110_02.dat
 SKIPPED: KV_101110_02.dat (no metadata for institute KIST)

Processing: KV_101112_04.dat
 SKIPPED: KV_101112_04.dat (no metadata for institute KIST)

Processing: KV_101112_05.dat
 SKIPPED: KV_101112_05.dat (no metadata for institute KIST)

Processing: KV_101112_06.dat
 SKIPPED: KV_101112_06.dat (no metadata f

In [ ]:
# ---------------------------------------------------------
# 1. Read and prepare the Excel metadata
# ---------------------------------------------------------
input_folder = r"C:\Users\mushimje\Desktop\M_MainFolder\EE_workspace\ETH\WP_workspaces\WP2\db_workspace\Extra_Kivu_CTDs\EAWAG_KIST_CTDs_RBR\Raw_data"
metadata_file = r"C:\Users\mushimje\Desktop\M_MainFolder\EE_workspace\ETH\WP_workspaces\WP2\db_workspace\Extra_Kivu_CTDs\EAWAG_KIST_CTDs_RBR\Metadata\Kivu_CTD_database_ag.xlsx"
extra_metadata_file = r"C:\Users\mushimje\Desktop\M_MainFolder\EE_workspace\ETH\WP_workspaces\WP2\db_workspace\Extra_Kivu_CTDs\EAWAG_KIST_CTDs_RBR\Metadata\Oct_2010_positions.xlsx"
which_institute = "EAWAG" # or KIST
process_extra_rbr_data(input_folder, metadata_file, extra_metadata_file, which_institute, year=None)


Processing: KV_100116_01.dat
Original metadata records: 11
Detected profiles: 11
Original metadata matches the detected number of profiles.
  Saved: KV_100116_01_A1.dat
  Saved: KV_100116_01_A2.dat
  Saved: KV_100116_01_A3.dat
  Saved: KV_100116_01_A4.dat
  Saved: KV_100116_01_A5.dat
  Saved: KV_100116_01_A6.dat
  Saved: KV_100116_01_A7.dat
  Saved: KV_100116_01_A8.dat
  Saved: KV_100116_01_A9.dat
  Saved: KV_100116_01_A10.dat
  Saved: KV_100116_01_A11.dat

Processing: KV_100117_01.dat
Original metadata records: 12
Detected profiles: 12
Original metadata matches the detected number of profiles.
  Saved: KV_100117_01_B1.dat
  Saved: KV_100117_01_B2.dat
  Saved: KV_100117_01_B3.dat
  Saved: KV_100117_01_B4.dat
  Saved: KV_100117_01_B5.dat
  Saved: KV_100117_01_B6.dat
  Saved: KV_100117_01_B7.dat
  Saved: KV_100117_01_B8.dat
  Saved: KV_100117_01_B9.dat
  Saved: KV_100117_01_B10.dat
  Saved: KV_100117_01_B11.dat
  Saved: KV_100117_01_B12.dat

Processing: KV_100118_01.dat
Original metadat

C:\Users\mushimje\AppData\Roaming\Python\Python311\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)
C:\Users\mushimje\AppData\Local\Temp\ipykernel_31864\1716997919.py:416: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Z1' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  recovered_records.loc[


  Saved: KV_101028_01_3.dat
  Saved: KV_101028_01_4.dat
  Saved: KV_101028_01_5.dat
  Saved: KV_101028_01_6.dat
  Saved: KV_101028_01_7.dat
  Saved: KV_101028_01_8.dat
  Saved: KV_101028_01_9.dat
  Saved: KV_101028_01_10.dat
  Saved: KV_101028_01_11.dat
  Saved: KV_101028_01_12.dat
  Saved: KV_101028_01_13.dat

Processing: KV_101108_04.dat
Original metadata records: 1
  SKIPPED: KV_101108_04.dat
  Reason: MISSING DEPTH COLUMN: No additional numerical column was available to recover Depth.

Processing: KV_101108_05.dat
Original metadata records: 1
  SKIPPED: KV_101108_05.dat
  Reason: MISSING DEPTH COLUMN: No additional numerical column was available to recover Depth.

Processing: KV_101108_06.dat
Original metadata records: 1
Detected profiles: 2
Original metadata is incomplete. Attempting recovery using extra metadata...
  SKIPPED: KV_101108_06.dat
  Reason: No matching records found in extra metadata for source filename: 'kp102502'.

Processing: KV_101109_02.dat
Original metadata reco